In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sqlite3
from scipy.stats import ttest_ind
import scipy.stats as stats
warnings.filterwarnings('ignore')

### Loading the included dataset
FY2025 reporting view. The aggregate export contains no transaction dates; the reporting label does not establish observed annual coverage.

In [ ]:
# Load the included aggregate snapshot; no external database is required.
df = pd.read_csv("dashboard/vendor_sales_summary_export.csv")
for column in ["VendorNumber", "Brand", "TotalPurchaseQuantity"]:
    df[column] = df[column].astype("int64")
df.head()

#### **Exploratory Data Analysis**

- Previously, we examined the various tables in the database to identify key variables, understand their relationships, and determine which ones should be included in the final analysis.

- In this phase of EDA, we will analyze the resultant table to gain insights into the distribution of each column. This will help us understand data patterns, identify anomalies, and ensure data quality before proceeding with further analysis.

In [ ]:
# Summary Statistics
df.describe().T

In [ ]:
# Distribution Plots for Numerical Columns 
numerical_cols= df.select_dtypes(include=np.number).columns

plt.figure(figsize=(15,10))
for i, col in enumerate(numerical_cols):
    plt.subplot(4,4,i+1)
    sns.histplot(df[col],kde=True, bins=30)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Outliers Detection with Boxplots
plt.figure(figsize=(15,10))
for i, col in enumerate(numerical_cols):
    plt.subplot(4,4,i+1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

#### **Summary Statistics Insights:** 
**Negative & Zero Values:**
- Gross Profit: Minimum value is -52,002.78, indicating losses. Some products or transactions may be selling at a loss due to high costs or selling at discounts lower than the purchase price.
- Profit Margin: Has a minimum of-00, which suggests cases where revenue is zero or even lower than costs.
- Total Sales Quantity & Sales Dollars: Minimum values are 0, meaning some products were purchased but never sold. These could be slow-moving or obsolet stock. 

**Outliers Indicated by High Standard Deviations:**
- Purchase & Actual Prices: The max values (5,681.81 & 7,499.99) are significantly higher than the mean (24.39 & 35.64), indicating potential premium products. 
- Freight Cost: Huge variation, from 0.09 to 257,032.07, suggests logistics inefficiencies or bulk shipments. 
- Stock Turnover: Ranges from 0 to 274.5, implying some products sell extremely fast while others remain in stock indefinitely. Value more than 1 indicates that Sold quantity for that product is higher than purchased quantity due to either sales are being fulfilled from older stock.

In [ ]:
# Match the dashboard population: profitable products with positive sales.
df = df[(df["GrossProfit"] > 0) & (df["ProfitMargin"] > 0) & (df["TotalSalesQuantity"] > 0)].copy()

In [ ]:
df

In [ ]:
# Count Plots for Categorical Columns
categorical_cols=["VendorName","Description"]

plt.figure(figsize=(12,5))
for i, col in enumerate(categorical_cols):
    plt.subplot(1,2,i+1)
    sns.countplot(y=df[col],order=df[col].value_counts().index[:10]) # Top 10 categories
    plt.title(f"Count Plot of {col}")
plt.tight_layout()
plt.show()

In [ ]:
# correlation Heatmap
plt.figure(figsize=(12,8))
correlation_matrix=df[numerical_cols].corr()
sns.heatmap(correlation_matrix,annot=True,fmt='.2f',cmap="coolwarm",linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

#### **Correlation Insights** 
- PurchasePrice has weak correlations with TotalSalesDollars (-0.012) and GrossProfit (-0.016), suggesting that price variations do not significantly impact sales revenue or profit. 
- Strong correlation between total purchase quantity and total sales quantity (0.999), confirming efficient inventory turnover. 
- Negative correlation between profit margin & total sales price (-0.179) suggests that as sales price increases, margins decrease, possibly due to competitive pricing pressures. 
- Stock Turnover has weak negative correlations with both GrossProfit (-0.038) and ProfitMargin (-0.055), indicating that faster turnover does not necessarily result in higher profitability

#### **Data Analysis**
**Identify Brands that needs Promotional or Pricing Adjustments which exhibit lower sales performance but higher profit margins**

In [ ]:
brand_performance= df.groupby('Description').agg({
    'TotalSalesDollars':'sum',
    'ProfitMargin':'mean'
}).reset_index()

In [ ]:
lower_sales_threshold=brand_performance['TotalSalesDollars'].quantile(0.15)
higher_margin_threshold=brand_performance['ProfitMargin'].quantile(0.85)

In [ ]:
lower_sales_threshold

In [ ]:
higher_margin_threshold

In [ ]:
# Filter brands with low sales but high profit margins
target_brands=brand_performance[
    (brand_performance['TotalSalesDollars']<=lower_sales_threshold)&
    (brand_performance['ProfitMargin']>=higher_margin_threshold)
]
print("Brands with Low Sales but High Profit Margins:")
display(target_brands.sort_values('TotalSalesDollars'))

In [ ]:
brand_performance=brand_performance[brand_performance['TotalSalesDollars']<10000] # for better visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=brand_performance, x='TotalSalesDollars', y='ProfitMargin', color="blue", label="All Brands", alpha = 0.2) 
sns.scatterplot(data=target_brands, x='TotalSalesDollars', y='ProfitMargin', color="red", label="Target Brands") 

plt.axhline(higher_margin_threshold, linestyle='--', color='black', label="High Margin Threshold") 
plt.axvline(lower_sales_threshold, linestyle='--', color='black', label="Low Sales Threshold") 

plt.xlabel("Total Sales ($)") 
plt.ylabel("Profit Margin (%)") 
plt.title("Brands for Promotional or Pricing Adjustments") 
plt.legend() 
plt.grid(True) 
plt.show()

**Which vendor and brands demonstrate the highest sales performance?**

In [ ]:
def format_dollars(value):
    if value>=1_000_000:
        return f"{value/ 1_000_000:.2f}M"
    elif value >=1_000:
        return f"{value/1_000:.2f}K"
    else:
        return str(value)

In [ ]:
# Top Vendors & Brands by Sales Performance
top_vendors=df.groupby("VendorName")["TotalSalesDollars"].sum().nlargest(10)
top_brands=df.groupby("Description")["TotalSalesDollars"].sum().nlargest(10)
top_vendors

In [ ]:
top_brands

In [ ]:
top_brands.apply(lambda x:format_dollars(x))

In [ ]:
top_vendors.apply(lambda x:format_dollars(x))

In [ ]:
plt.figure(figsize=(15,5))

#Plot for Top Vendors
plt.subplot(1, 2, 1)
ax1 = sns.barplot(y=top_vendors.index, x=top_vendors.values, palette="Blues_r") 
plt.title("Top 10 Vendors by Sales")

for bar in ax1.patches:
    ax1.text(bar.get_width() + (bar.get_width() * 0.02), 
             bar.get_y() + bar.get_height() / 2, 
             format_dollars(bar.get_width()), 
             ha='left', va='center', fontsize=10, color='black')

#Plot for Top Brand

plt.subplot(1, 2, 2)
ax2 = sns.barplot(y=top_brands.index.astype(str), x=top_brands.values, palette="Reds_r") 
plt.title("Top 10 Brands by Sales")
for bar in ax2.patches:
    ax2.text(bar.get_width() + (bar.get_width() * 0.02),
             bar.get_y() + bar.get_height() / 2, 
             format_dollars(bar.get_width()), 
             ha='left', va='center', fontsize=10, color='black')

plt.tight_layout()
plt.show()

**Which vendor contribute the most to total purchase dollars?**

In [ ]:
vendor_performance=df.groupby('VendorName').agg({
    'TotalPurchaseDollars':'sum',
    'GrossProfit':'sum',
    'TotalSalesDollars':'sum'
}).reset_index()

In [ ]:
vendor_performance['PurchaseContribution%']=(vendor_performance['TotalPurchaseDollars']/vendor_performance['TotalPurchaseDollars'].sum())*100

In [ ]:
vendor_performance=round(vendor_performance.sort_values(by='PurchaseContribution%',ascending=False),2)

In [ ]:
# Display Top 10 Vendors
top_vendors = vendor_performance.head(10) 
top_vendors['TotalSalesDollars'] = top_vendors['TotalSalesDollars'].apply(format_dollars) 
top_vendors['TotalPurchaseDollars'] = top_vendors['TotalPurchaseDollars'].apply(format_dollars) 
top_vendors['GrossProfit'] = top_vendors['GrossProfit'].apply(format_dollars) 
top_vendors

In [ ]:
top_vendors['Cumulative_Contribution%']=top_vendors['PurchaseContribution%'].cumsum()
top_vendors

In [ ]:
fig, ax1= plt.subplots(figsize=(10, 6)) 

#Bar plot for Purchase Contribution% 
sns.barplot(x=top_vendors['VendorName'], y=top_vendors['PurchaseContribution%'], palette="mako", ax=ax1) 
for i, value in enumerate(top_vendors['PurchaseContribution%']): 
    ax1.text(i, value-1, str(value)+'%', ha='center', fontsize=10, color='white') 
    
# Line Plot for Cumulative Contribution% 
ax2=ax1.twinx() 
ax2.plot(top_vendors['VendorName'], top_vendors['Cumulative_Contribution%'], color='red', marker='o', linestyle='dashed', label='Cumulative%')

ax1.set_xticklabels (top_vendors['VendorName'], rotation=90) 
ax1.set_ylabel('Purchase Contribution %', color='blue') 
ax2.set_ylabel('Cumulative Contribution%', color='red') 
ax1.set_xlabel('Vendors') 
ax1.set_title('Pareto Chart: Vendor Contribution to Total Purchases') 
ax2.axhline(y=100, color='gray', linestyle='dashed', alpha=0.7) 
ax2.legend(loc='upper right')

plt.show()

**How much of total procurement is dependent on the top vendors?**

In [ ]:
print(f"Total Purchase Contribution of top 10 vendors is {round(top_vendors['PurchaseContribution%'].sum(),2)} %")

In [ ]:
vendors = list(top_vendors ['VendorName'].values)
purchase_contributions = list(top_vendors['PurchaseContribution%'].values)
total_contribution = sum(purchase_contributions)
remaining_contribution = 100-total_contribution

#Append "Other Vendors" category

vendors.append("Other Vendors")
purchase_contributions.append(remaining_contribution)

#Donut Chart
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(purchase_contributions, labels=vendors, autopct='%1.1f%%',
                                   startangle=140, pctdistance=0.85, colors=plt.cm.Paired.colors)

#Draw a white circle in the center to create a "donut" effect

centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig.gca().add_artist(centre_circle)

#Add Total Contribution annotation in the center

plt.text(0, 0, f"Top 10 Total:\n{total_contribution:.2f}%", fontsize=14, fontweight='bold', ha='center',va='center')

plt.title("Top 10 Vendor's Purchase Contribution(%)")
plt.show()
         

**Does purchasing in bulk reduce the unit price and what is the optimal purchase volume for cost savings?**

In [ ]:
df['UnitPurchasePrice']=df['TotalPurchaseDollars']/df['TotalPurchaseQuantity']

In [ ]:
df["OrderSize"]=pd.qcut(df["TotalPurchaseQuantity"],q=3,labels=["Small","Medium","Large"])

In [ ]:
df.groupby("OrderSize")[['UnitPurchasePrice']].mean()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df,x="OrderSize",y="UnitPurchasePrice",palette="Set2")
plt.title("Impact of Bulk Purchasing on Unit Price")
plt.xlabel("Order Size")
plt.ylabel("Average Unit Purchase Price")
plt.show()

- Vendors buying in bulk (Large Order Size) get the lowest unit price ($10.78 per unit), meaning higher margins if they can manage inventory efficiently.

- The price difference between Small and Large orders is substantial (~72% reduction in unit cost)

- This suggests that bulk pricing strategies successfully encourage vendors to purchase in larger volumes, leading to higher overall sales despite lower per-unit revenue.

**Which vendors have low inventory turnover, indicating excess stock and slow-moving products?**

In [ ]:
df[df['StockTurnover']<1].groupby('VendorName')[['StockTurnover']].mean().sort_values(by='StockTurnover' ,ascending=True).head(10)

**How Much capital is locked in unsold inventory per vendor, and which vendors contribute the most to it?**

In [ ]:
df['UnsoldInventoryValue']=(df['TotalPurchaseQuantity']-df['TotalSalesQuantity'])*df['PurchasePrice']
print('Total Unsold Capital:',format_dollars(df['UnsoldInventoryValue'].sum()))

In [ ]:
# Aggregate Capital Locked per Vendor
inventory_value_per_vendor = df.groupby("VendorName")["UnsoldInventoryValue"].sum().reset_index()

#Sort Vendors with the Highest Locked Capital

inventory_value_per_vendor = inventory_value_per_vendor.sort_values(by="UnsoldInventoryValue", ascending=False)
inventory_value_per_vendor['UnsoldInventoryValue'] = inventory_value_per_vendor['UnsoldInventoryValue'].apply(format_dollars)
inventory_value_per_vendor.head(10)

**What is the 95% confidence intervals for profit margins of top-performing and low-performing vendors.**

In [ ]:
top_threshold=df["TotalSalesDollars"].quantile(0.75)
low_threshold=df["TotalSalesDollars"].quantile(0.25)

In [ ]:
top_vendors=df[df["TotalSalesDollars"]>=top_threshold]["ProfitMargin"].dropna()
low_vendors=df[df["TotalSalesDollars"]<=low_threshold]["ProfitMargin"].dropna()

In [ ]:
def confidence_interval(data, confidence=0.95):
    mean_val = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(len(data)) # Standard error
    t_critical = stats.t.ppf((1 + confidence) / 2, df=len(data) - 1)
    margin_of_error = t_critical*std_err
    return mean_val, mean_val - margin_of_error, mean_val + margin_of_error

In [ ]:
top_mean, top_lower, top_upper = confidence_interval(top_vendors)
low_mean, low_lower, low_upper = confidence_interval(low_vendors)

print(f"Top Vendors 95% CI: ({top_lower:.2f}, {top_upper:.2f}), Mean: {top_mean:.2f}")
print(f"Low Vendors 95% CI: ({low_lower:.2f}, {low_upper:.2f}), Mean: {low_mean:.2f}")

plt.figure(figsize=(12,6))

#Top Vendors Plot

sns.histplot(top_vendors, kde=True, color="blue", bins=30, alpha=0.5, label="Top Vendors")
plt.axvline(top_lower, color="blue", linestyle="--", label=f"Top Lower: {top_lower:.2f}")
plt.axvline(top_upper, color="blue", linestyle="--", label=f"Top Upper: {top_upper:.2f}")
plt.axvline(top_mean, color="blue", linestyle="-", label=f"Top Mean: {top_mean:.2f}")

# Low Vendors Plot

sns.histplot(low_vendors, kde=True, color="red", bins=30, alpha=0.5, label="Low Vendors")
plt.axvline(low_lower, color="red", linestyle="--", label=f"Low Lower: {low_lower:.2f}")
plt.axvline(low_upper, color="red", linestyle="--", label=f"Low Upper: {low_upper:.2f}")
plt.axvline(low_mean, color="red", linestyle="-", label=f"Low Mean: {low_mean:.2f}")

# Finalize Plot

plt.title("Confidence Interval Comparison: Top vs. Low Vendors (Profit Margin)")
plt.xlabel("Profit Margin (%)")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()

The lower-sales product group has mean margin 41.57% (95% interval 40.50–42.64%), compared with 31.18% (30.74–31.61%) for the high-sales group. These are descriptive product-record groups; shared vendors can affect independence.


**Is there a significant difference in profit margins between top-performing and low-performing vendors?**

Hypothesis:

H₀(Null Hypothesis): There is no significant difference in the mean profit margins of top-performing and low-performing vendors.

H₁ (Alternative Hypothesis): The mean profit margins of top-performing and low-performing vendors are significantly different.

In [ ]:
top_threshold = df ["TotalSalesDollars"].quantile(0.75)
low_threshold = df ["TotalSalesDollars"].quantile (0.25)

top_vendors = df [df ["TotalSalesDollars"] >= top_threshold] ["ProfitMargin"].dropna()
low_vendors = df [df ["TotalSalesDollars"] <= low_threshold] ["ProfitMargin"].dropna()

#Perform Two-Sample T-Test

t_stat, p_value = ttest_ind(top_vendors, low_vendors, equal_var=False)

#Print results

print(f"T-Statistic: {t_stat:.4f}, P-Value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject Ho: There is a significant difference in profit margins between top and low-performing vendors.")
else:
    print("Fail to Reject Ho: No significant difference in profit margins.")